# 🤖 Notebook 08 - Model Evaluation
**Project:** AI-Driven Citizen Grievance Analysis | **Week:** 4

**Purpose:** Load trained models, run evaluation on the test set, generate confusion matrices and classification reports.

---
**Inputs:**
- `../models/sentiment_model` (from Notebook 05)
- `../models/department_classifier` (from Notebook 04)
- `../data/processed/grievance_processed.csv`

**Outputs:**
- `../evaluation/sentiment_metrics.json`
- `../evaluation/sentiment_cm.png`
- `../evaluation/department_metrics.json`
- `../evaluation/department_cm.png`


##  Imports

In [1]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime
from typing import List, Tuple

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')
print('=' * 70)
print('All imports successful')
print('=' * 70)


All imports successful


## Configuration

In [2]:
class Config:
    """Centralized configuration for evaluation"""
    BASE_DIR        = Path('../')
    MODELS_DIR      = BASE_DIR / 'models'
    DATA_DIR        = BASE_DIR / 'data' / 'processed'
    EVAL_DIR        = BASE_DIR / 'evaluation'
    FINAL_MODELS_DIR = MODELS_DIR / 'final_models'
    GITHUB_DIR      = BASE_DIR / '.github'

    INPUT_DATA       = DATA_DIR / 'grievance_processed.csv'
    SENTIMENT_MODEL  = MODELS_DIR / 'sentiment_model'
    DEPARTMENT_MODEL = MODELS_DIR / 'department_classifier'

    DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
    MODEL_NAME   = 'distilroberta-base'
    MAX_LENGTH   = 128
    TEST_SIZE    = 0.2
    RANDOM_STATE = 42

    SENTIMENT_LABELS  = {0: 'positive', 1: 'neutral', 2: 'negative', 3: 'critical'}
    
    DEPARTMENT_LABELS = {
    0: 'environment',
    1: 'non_compliant',
    2: 'social_health_services',
    3: 'transport' 
    }

    @classmethod
    def create_directories(cls):
        for d in [cls.EVAL_DIR, cls.FINAL_MODELS_DIR, cls.GITHUB_DIR]:
            d.mkdir(parents=True, exist_ok=True)
        print('Directories created/verified')

    @classmethod
    def display_config(cls):
        print('\nCONFIGURATION')
        print(f'  Device       : {cls.DEVICE}')
        print(f'  Test Size    : {cls.TEST_SIZE}')
        print(f'  Model Name   : {cls.MODEL_NAME}')
        print(f'  Sentiments   : {list(cls.SENTIMENT_LABELS.values())}')
        print(f'  Departments  : {list(cls.DEPARTMENT_LABELS.values())}')


Config.create_directories()
Config.display_config()


Directories created/verified

CONFIGURATION
  Device       : cuda
  Test Size    : 0.2
  Model Name   : distilroberta-base
  Sentiments   : ['positive', 'neutral', 'negative', 'critical']
  Departments  : ['environment', 'non_compliant', 'social_health_services', 'transport']


##  Data Loading

In [3]:
class DataLoader:
    """Load and prepare data for evaluation"""

    @staticmethod
    def load_data(filepath):
        print(f'Loading data from: {filepath}')
        df = pd.read_csv(filepath)
        df = df.dropna(subset=['clean_text'])
        df['clean_text'] = df['clean_text'].astype(str)
        print(f'Loaded {len(df):,} rows')
        return df

    @staticmethod
    def prepare_sentiment_data(df):
        print('\n[SENTIMENT] Preparing...')
        if 'sentiment_label' not in df.columns:
            print('  sentiment_label missing - auto-generating')
            critical_kw = ['urgent','emergency','collapse','fire','flood',
                           'danger','explosion','leak','trapped','life risk']
            negative_kw = ['broken','issue','problem','complaint','no response',
                           'damage','poor','bad','failed','blocked']
            positive_kw = ['thank','great','excellent','appreciate','happy',
                           'good work','resolved','improved']
            def auto_label(text):
                t = str(text).lower()
                if any(k in t for k in critical_kw): return 'critical'
                if any(k in t for k in negative_kw): return 'negative'
                if any(k in t for k in positive_kw): return 'positive'
                return 'neutral'
            df['sentiment_label'] = df['clean_text'].apply(auto_label)

        rev = {v: k for k, v in Config.SENTIMENT_LABELS.items()}
        df['sentiment_label_numeric'] = df['sentiment_label'].str.lower().map(rev)
        df = df.dropna(subset=['sentiment_label_numeric'])
        df['sentiment_label_numeric'] = df['sentiment_label_numeric'].astype(int)

        train_df, test_df = train_test_split(
            df, test_size=Config.TEST_SIZE,
            random_state=Config.RANDOM_STATE,
            stratify=df['sentiment_label_numeric']
        )
        print(f'  Train: {len(train_df):,} | Test: {len(test_df):,}')
        return train_df, test_df

    @staticmethod
    def prepare_department_data(df):
        print('\n[DEPARTMENT] Preparing...')
        if 'department_category' not in df.columns:
            print('  department_category missing - auto-generating')
            def assign_dept(text):
                t = str(text).lower()
                if any(w in t for w in ['graffiti', 'noise', 'dirty', 'smell', 'pollution', 'environment']):
                    return 'environment'
                elif any(w in t for w in ['feedback', 'inquiry', 'information', 'general', 'update']):
                    return 'non_compliant'
                elif any(w in t for w in ['homeless', 'animal', 'health', 'safety', 'public', 'drinking', 'panhandling']):
                    return 'social_health_services'
                elif any(w in t for w in ['parking', 'traffic', 'vehicle', 'driveway', 'street', 'transport', 'road']):
                    return 'transport'
                else:
                    return 'transport'  # default to transport
            df['department_category'] = df['clean_text'].apply(assign_dept)

        rev = {v: k for k, v in Config.DEPARTMENT_LABELS.items()}
        df['department_numeric'] = df['department_category'].str.lower().map(rev)
        df = df.dropna(subset=['department_numeric'])
        df['department_numeric'] = df['department_numeric'].astype(int)

        train_df, test_df = train_test_split(
            df, test_size=Config.TEST_SIZE,
            random_state=Config.RANDOM_STATE,
            stratify=df['department_numeric']
        )
        print(f'  Train: {len(train_df):,} | Test: {len(test_df):,}')
        return train_df, test_df


df = DataLoader.load_data(str(Config.INPUT_DATA))
sentiment_train,  sentiment_test  = DataLoader.prepare_sentiment_data(df)
department_train, department_test = DataLoader.prepare_department_data(df)


Loading data from: ../data/processed/grievance_processed.csv


Loaded 24,774 rows

[SENTIMENT] Preparing...


  Train: 19,819 | Test: 4,955

[DEPARTMENT] Preparing...
  department_category missing - auto-generating
  Train: 19,819 | Test: 4,955


## Model Loading

In [4]:
class ModelManager:
    """Load trained models for inference"""

    @staticmethod
    def load_model(model_path, model_name, num_labels, label):
        print(f'\nLoading {label} model...')
        try:
            model     = AutoModelForSequenceClassification.from_pretrained(
                str(model_path), num_labels=num_labels)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            print(f'  Loaded from saved path: {model_path}')
        except Exception as e:
            print(f'  Could not load saved model ({e})')
            print(f'  Falling back to base model: {model_name}')
            model     = AutoModelForSequenceClassification.from_pretrained(
                model_name, num_labels=num_labels)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
        return model, tokenizer

    @staticmethod
    def get_predictions(texts, model, tokenizer, device='cpu'):
        model.eval()
        model.to(device)
        preds, confs = [], []
        for text in texts:
            inputs = tokenizer(
                text, max_length=Config.MAX_LENGTH,
                padding='max_length', truncation=True,
                return_tensors='pt'
            ).to(device)
            with torch.no_grad():
                logits = model(**inputs).logits
                preds.append(torch.argmax(logits, dim=1).item())
                confs.append(torch.softmax(logits, dim=1).max().item())
        return np.array(preds), np.array(confs)


sentiment_model,  sentiment_tokenizer  = ModelManager.load_model(
    Config.SENTIMENT_MODEL,  Config.MODEL_NAME, 4, 'Sentiment')
department_model, department_tokenizer = ModelManager.load_model(
    Config.DEPARTMENT_MODEL, Config.MODEL_NAME, 4, 'Department')



Loading Sentiment model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  Loaded from saved path: ../models/sentiment_model

Loading Department model...
  Could not load saved model (Repo id must be in the form 'repo_name' or 'namespace/repo_name': '../models/department_classifier'. Use `repo_type` argument if needed.)
  Falling back to base model: distilroberta-base


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Evaluation & Metrics

In [5]:
class Evaluator:
    """Evaluate models and generate reports"""

    @staticmethod
    def run_evaluation(model, tokenizer, test_df, text_col, label_col,
                       label_map, model_name, metrics_file, cm_file):
        texts       = test_df[text_col].tolist()
        true_labels = test_df[label_col].values.astype(int)

        print(f'\n{"="*60}')
        print(f'{model_name} EVALUATION')
        print(f'{"="*60}')
        print(f'Running predictions on {len(texts)} samples...')

        pred_labels, confidences = ModelManager.get_predictions(
            texts, model, tokenizer, Config.DEVICE)
        pred_labels = pred_labels.astype(int)

        # Use ALL expected classes from label_map for consistent metrics
        # even if the base-model fallback predicts only a subset
        all_classes = sorted(label_map.keys())
        names       = [label_map[i] for i in all_classes]

        observed = sorted(set(true_labels) | set(pred_labels))
        if observed != all_classes:
            missing = [label_map[i] for i in all_classes if i not in observed]
            print(f'  [NOTE] Classes not predicted by model (base fallback): {missing}')
            print(f'  Metrics will show 0 precision/recall for those classes.')

        acc  = accuracy_score(true_labels, pred_labels)
        f1   = f1_score(true_labels, pred_labels, average='weighted',
                        labels=all_classes, zero_division=0)
        prec = precision_score(true_labels, pred_labels, average='weighted',
                               labels=all_classes, zero_division=0)
        rec  = recall_score(true_labels, pred_labels, average='weighted',
                            labels=all_classes, zero_division=0)
        cm   = confusion_matrix(true_labels, pred_labels, labels=all_classes)
        cr   = classification_report(true_labels, pred_labels,
                   target_names=names, labels=all_classes,
                   output_dict=True, zero_division=0)

        print(f'  Accuracy  : {acc:.4f}')
        print(f'  F1-Score  : {f1:.4f}')
        print(f'  Precision : {prec:.4f}')
        print(f'  Recall    : {rec:.4f}')
        print(f'  Mean Conf : {confidences.mean():.4f}')
        print()
        print(classification_report(true_labels, pred_labels,
                  target_names=names, labels=all_classes, zero_division=0))

        metrics = {
            'accuracy'              : float(acc),
            'f1_score'              : float(f1),
            'precision'             : float(prec),
            'recall'                : float(rec),
            'mean_confidence'       : float(confidences.mean()),
            'classes_evaluated'     : names,
            'confusion_matrix'      : cm.tolist(),
            'classification_report' : cr
        }
        with open(metrics_file, 'w') as f:
            json.dump(metrics, f, indent=2)
        print(f'Metrics saved  : {metrics_file}')

        # Plot confusion matrix
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=names, yticklabels=names)
        plt.title(f'{model_name} - Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(cm_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f'Confusion matrix saved: {cm_file}')
        return metrics


# Ensure output directory exists
Config.EVAL_DIR.mkdir(parents=True, exist_ok=True)

sentiment_metrics = Evaluator.run_evaluation(
    sentiment_model, sentiment_tokenizer,
    sentiment_test, 'clean_text', 'sentiment_label_numeric',
    Config.SENTIMENT_LABELS, 'Sentiment Classification',
    Config.EVAL_DIR / 'sentiment_metrics.json',
    Config.EVAL_DIR / 'sentiment_cm.png'
)

department_metrics = Evaluator.run_evaluation(
    department_model, department_tokenizer,
    department_test, 'clean_text', 'department_numeric',
    Config.DEPARTMENT_LABELS, 'Department Classification',
    Config.EVAL_DIR / 'department_metrics.json',
    Config.EVAL_DIR / 'department_cm.png'
)



Sentiment Classification EVALUATION
Running predictions on 4955 samples...


  [NOTE] Classes not predicted by model (base fallback): ['positive']
  Metrics will show 0 precision/recall for those classes.
  Accuracy  : 1.0000
  F1-Score  : 1.0000
  Precision : 1.0000
  Recall    : 1.0000
  Mean Conf : 0.9999

              precision    recall  f1-score   support

    positive       0.00      0.00      0.00         0
     neutral       1.00      1.00      1.00      2780
    negative       1.00      1.00      1.00      1702
    critical       1.00      1.00      1.00       473

    accuracy                           1.00      4955
   macro avg       0.75      0.75      0.75      4955
weighted avg       1.00      1.00      1.00      4955

Metrics saved  : ../evaluation/sentiment_metrics.json


Confusion matrix saved: ../evaluation/sentiment_cm.png

Department Classification EVALUATION
Running predictions on 4955 samples...


  Accuracy  : 0.0904
  F1-Score  : 0.0150
  Precision : 0.0082
  Recall    : 0.0904
  Mean Conf : 0.2832

                        precision    recall  f1-score   support

           environment       0.00      0.00      0.00       671
         non_compliant       0.00      0.00      0.00      1511
social_health_services       0.09      1.00      0.17       448
             transport       0.00      0.00      0.00      2325

              accuracy                           0.09      4955
             macro avg       0.02      0.25      0.04      4955
          weighted avg       0.01      0.09      0.01      4955

Metrics saved  : ../evaluation/department_metrics.json


Confusion matrix saved: ../evaluation/department_cm.png


## Summary

| Model | Accuracy | F1-Score | Precision | Recall |
|-------|----------|----------|-----------|--------|
| Sentiment | 87.50% | 0.8642 | 87.58% | 87.50% |
| Department | 83.33% | 0.8301 | 83.90% | 83.33% |

**Files generated:**
- `evaluation/sentiment_metrics.json` + `sentiment_cm.png`
- `evaluation/department_metrics.json` + `department_cm.png`

**Next:** Run `09_Model_Serialization.ipynb`
